# Platform Services for AI Development

This notebook introduces the Thinkube platform services and validates they're working.

**Goal**: Understand how these services combine to build the AI Research Assistant.

## The AI Application Stack

Building an AI Research Assistant requires several interconnected services:

```
┌─────────────────────────────────────────────────────────────────┐
│                     AI Research Assistant                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   [ArXiv Papers] ──► [Chunk] ──► [Embed] ──► [Store Vectors]    │
│                                     │              │             │
│                                     ▼              ▼             │
│                                 LiteLLM         Qdrant           │
│                                     │              │             │
│   [User Query] ──► [Embed] ──► [Search] ◄────────┘             │
│                        │           │                             │
│                        ▼           ▼                             │
│                    LiteLLM    [Relevant Chunks]                  │
│                                    │                             │
│                                    ▼                             │
│                         [Generate Answer] ◄─── LiteLLM          │
│                                    │                             │
│                                    ▼                             │
│                              [Response]                          │
│                                    │                             │
│         ┌──────────────────────────┼──────────────────────┐     │
│         ▼                          ▼                      ▼     │
│     Langfuse                  PostgreSQL               Valkey   │
│   (trace calls)            (paper metadata)          (cache)    │
│                                                                  │
│   Multi-Agent Coordination: NATS messaging                      │
│   Experiment Tracking: MLflow                                   │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

Let's validate each service and understand its role.

In [ ]:
# Helper functions for colored output
from IPython.display import display, HTML

def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✅ {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">❌ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ️  {msg}</span>'))

def skip(msg):
    display(HTML(f'<span style="color: #95a5a6;">⊘ {msg}</span>'))

---
## 1. LiteLLM - The LLM Gateway

**What it does**: Provides a unified API to access multiple LLM providers (OpenAI, Anthropic, local models).

**Why we need it**: 
- Single API for all LLMs - switch models without code changes
- Cost tracking and rate limiting
- Route between local and cloud models

**In the Research Assistant**:
- Generate embeddings for papers
- Answer questions about research
- Power the multi-agent system

In [ ]:
import os

litellm_endpoint = os.environ.get('LITELLM_ENDPOINT')
litellm_key = os.environ.get('LITELLM_MASTER_KEY')

if not litellm_endpoint or not litellm_key:
    skip("LiteLLM not configured - set LITELLM_ENDPOINT and LITELLM_MASTER_KEY")
else:
    try:
        from openai import OpenAI
        
        # LiteLLM provides an OpenAI-compatible API
        client = OpenAI(
            base_url=litellm_endpoint,
            api_key=litellm_key
        )
        
        # List available models
        models = client.models.list()
        model_names = [m.id for m in models.data][:5]
        
        success(f"Connected to LiteLLM at {litellm_endpoint}")
        info(f"Available models: {', '.join(model_names)}{'...' if len(models.data) > 5 else ''}")
        
        # Test a simple completion
        if model_names:
            response = client.chat.completions.create(
                model=model_names[0],
                messages=[{"role": "user", "content": "Say 'hello' in one word."}],
                max_tokens=10
            )
            info(f"Test completion: {response.choices[0].message.content}")
        
    except Exception as e:
        error(f"LiteLLM connection failed: {e}")

---
## 2. Qdrant - Vector Database for RAG

**What it does**: Stores and searches vector embeddings.

**Why we need it**:
- Semantic search - find papers by meaning, not just keywords
- Fast similarity search over millions of vectors
- Metadata filtering (by date, author, topic)

**In the Research Assistant**:
- Store embeddings of paper chunks
- Find relevant context for questions
- Power the RAG pipeline

In [ ]:
qdrant_url = os.environ.get('QDRANT_URL')

if not qdrant_url:
    skip("Qdrant not configured - set QDRANT_URL")
else:
    try:
        from qdrant_client import QdrantClient
        from qdrant_client.models import Distance, VectorParams, PointStruct
        import uuid
        
        client = QdrantClient(
            url=qdrant_url,
            port=443,
            https=True,
            verify=False
        )
        
        collections = client.get_collections()
        success(f"Connected to Qdrant at {qdrant_url}")
        info(f"Existing collections: {len(collections.collections)}")
        
        # Demo: Create a test collection, add vectors, search
        test_collection = "thinkube_test"
        
        # Create collection (if not exists)
        if not client.collection_exists(test_collection):
            client.create_collection(
                collection_name=test_collection,
                vectors_config=VectorParams(size=4, distance=Distance.COSINE)
            )
        
        # Insert test vectors (simulating paper embeddings)
        client.upsert(
            collection_name=test_collection,
            points=[
                PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={"title": "LoRA paper"}),
                PointStruct(id=2, vector=[0.2, 0.3, 0.4, 0.5], payload={"title": "QLoRA paper"}),
            ]
        )
        
        # Search for similar vectors
        results = client.search(
            collection_name=test_collection,
            query_vector=[0.15, 0.25, 0.35, 0.45],
            limit=2
        )
        
        info(f"Search test - found {len(results)} similar documents")
        for r in results:
            info(f"  - {r.payload['title']} (score: {r.score:.3f})")
        
        # Cleanup
        client.delete_collection(test_collection)
        
    except Exception as e:
        error(f"Qdrant connection failed: {e}")

---
## 3. Langfuse - LLM Observability

**What it does**: Traces LLM calls, tracks costs, monitors quality.

**Why we need it**:
- Debug agent behavior - see every LLM call
- Track costs across models
- Identify slow or failing requests
- Evaluate response quality over time

**In the Research Assistant**:
- Trace RAG pipeline execution
- Monitor multi-agent coordination
- Debug when answers are wrong

In [ ]:
langfuse_host = os.environ.get('LANGFUSE_HOST')
langfuse_public_key = os.environ.get('LANGFUSE_PUBLIC_KEY')
langfuse_secret_key = os.environ.get('LANGFUSE_SECRET_KEY')

if not all([langfuse_host, langfuse_public_key, langfuse_secret_key]):
    skip("Langfuse not configured - set LANGFUSE_HOST, LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY")
else:
    try:
        from langfuse import Langfuse
        
        langfuse = Langfuse(
            public_key=langfuse_public_key,
            secret_key=langfuse_secret_key,
            host=langfuse_host
        )
        
        # Verify authentication
        langfuse.auth_check()
        success(f"Connected to Langfuse at {langfuse_host}")
        
        # Demo: Create a trace (like we would for a RAG query)
        trace = langfuse.trace(
            name="research-assistant-test",
            metadata={"source": "platform-validation"}
        )
        
        # Simulate tracing a RAG pipeline
        span = trace.span(name="embed-query")
        span.end(output={"dimensions": 1536})
        
        span = trace.span(name="vector-search")
        span.end(output={"results": 5})
        
        span = trace.span(name="generate-answer")
        span.end(output={"tokens": 150})
        
        langfuse.flush()
        info(f"Created test trace - view at {langfuse_host}")
        
    except Exception as e:
        error(f"Langfuse connection failed: {e}")

---
## 4. MLflow - Experiment Tracking

**What it does**: Tracks ML experiments, stores models, manages deployments.

**Why we need it**:
- Track fine-tuning experiments (hyperparameters, metrics)
- Compare model versions
- Store and version trained models

**In the Research Assistant**:
- Track fine-tuning runs for the research domain
- Link papers to experiments ("this paper inspired experiment X")
- Compare base vs fine-tuned model performance

In [ ]:
mlflow_uri = os.environ.get('MLFLOW_TRACKING_URI')

if not mlflow_uri:
    skip("MLflow not configured - set MLFLOW_TRACKING_URI")
else:
    try:
        import mlflow
        import requests
        import urllib3
        urllib3.disable_warnings()
        
        # MLflow with Keycloak OAuth
        mlflow_username = os.environ.get('MLFLOW_AUTH_USERNAME')
        mlflow_password = os.environ.get('MLFLOW_AUTH_PASSWORD')
        token_url = os.environ.get('MLFLOW_KEYCLOAK_TOKEN_URL')
        client_id = os.environ.get('MLFLOW_KEYCLOAK_CLIENT_ID')
        client_secret = os.environ.get('MLFLOW_CLIENT_SECRET')
        
        if not all([mlflow_username, mlflow_password, token_url, client_id, client_secret]):
            skip("MLflow OAuth not fully configured")
        else:
            # Get OAuth token
            token_response = requests.post(
                token_url,
                data={
                    'grant_type': 'password',
                    'client_id': client_id,
                    'client_secret': client_secret,
                    'username': mlflow_username,
                    'password': mlflow_password
                },
                verify=False
            )
            
            if token_response.status_code == 200:
                mlflow_token = token_response.json().get('access_token')
                os.environ['MLFLOW_TRACKING_TOKEN'] = mlflow_token
                mlflow.set_tracking_uri(mlflow_uri)
                
                experiments = mlflow.search_experiments(max_results=5)
                success(f"Connected to MLflow at {mlflow_uri}")
                info(f"Experiments: {len(experiments)}")
                
                # Demo: Log a test run (like we would for fine-tuning)
                mlflow.set_experiment("research-assistant-validation")
                with mlflow.start_run(run_name="platform-test"):
                    mlflow.log_param("model", "test")
                    mlflow.log_metric("accuracy", 0.95)
                info("Created test experiment run")
            else:
                error(f"MLflow OAuth failed: {token_response.status_code}")
                
    except Exception as e:
        error(f"MLflow connection failed: {e}")

---
## 5. PostgreSQL - Relational Data

**What it does**: Traditional relational database for structured data.

**Why we need it**:
- Store paper metadata (title, authors, date, abstract)
- Track which papers have been processed
- User preferences and history
- Relational queries ("papers by author X in 2024")

**In the Research Assistant**:
- Paper catalog with metadata
- Processing status tracking
- Link papers to MLflow experiments

In [ ]:
postgres_host = os.environ.get('POSTGRES_HOST')
postgres_password = os.environ.get('POSTGRES_PASSWORD')

if not postgres_host or not postgres_password:
    skip("PostgreSQL not configured - set POSTGRES_HOST and POSTGRES_PASSWORD")
else:
    try:
        import psycopg2
        
        conn = psycopg2.connect(
            host=postgres_host,
            port=int(os.environ.get('POSTGRES_PORT', 5432)),
            database=os.environ.get('POSTGRES_DB', 'postgres'),
            user=os.environ.get('POSTGRES_USER', 'postgres'),
            password=postgres_password
        )
        
        cursor = conn.cursor()
        cursor.execute('SELECT version();')
        version = cursor.fetchone()[0]
        
        success(f"Connected to PostgreSQL at {postgres_host}")
        info(f"Version: {version.split(',')[0]}")
        
        # Demo: Show how we'd store paper metadata
        info("Example schema for Research Assistant:")
        info("  papers(id, arxiv_id, title, authors, abstract, published_date)")
        info("  chunks(id, paper_id, content, embedding_id, chunk_index)")
        info("  experiments(id, paper_id, mlflow_run_id, notes)")
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        error(f"PostgreSQL connection failed: {e}")

---
## 6. Valkey - Caching Layer

**What it does**: In-memory key-value store (Redis-compatible).

**Why we need it**:
- Cache expensive LLM calls
- Cache embeddings for repeated queries
- Session state for conversations
- Rate limiting

**In the Research Assistant**:
- Cache paper summaries (don't re-summarize the same paper)
- Cache embedding results
- Store conversation context

In [ ]:
valkey_host = os.environ.get('VALKEY_HOST')
valkey_port = os.environ.get('VALKEY_PORT')

if not valkey_host or not valkey_port:
    skip("Valkey not configured - set VALKEY_HOST and VALKEY_PORT")
else:
    try:
        import redis
        import json
        
        client = redis.Redis(
            host=valkey_host,
            port=int(valkey_port),
            decode_responses=True
        )
        
        client.ping()
        server_info = client.info('server')
        
        success(f"Connected to Valkey at {valkey_host}")
        info(f"Version: {server_info.get('redis_version', 'unknown')}")
        
        # Demo: Cache a paper summary (like we would in the Research Assistant)
        paper_id = "arxiv:2106.09685"  # LoRA paper
        cache_key = f"summary:{paper_id}"
        
        # Simulate caching a summary
        summary = {"title": "LoRA", "summary": "Low-rank adaptation for efficient fine-tuning"}
        client.setex(cache_key, 3600, json.dumps(summary))  # Cache for 1 hour
        
        # Retrieve from cache
        cached = json.loads(client.get(cache_key))
        info(f"Cache test - stored and retrieved: {cached['title']}")
        
        # Cleanup
        client.delete(cache_key)
        
    except Exception as e:
        error(f"Valkey connection failed: {e}")

---
## 7. NATS - Message Broker for Multi-Agent

**What it does**: High-performance messaging system for pub/sub and request/reply.

**Why we need it**:
- Decouple agents - they communicate via messages, not direct calls
- Scale agents independently
- Async processing - agents can work in parallel

**In the Research Assistant**:
- Paper Summarizer publishes summaries
- Experiment Tracker subscribes to new papers
- Insight Finder coordinates across agents

In [ ]:
nats_url = os.environ.get('NATS_URL')

if not nats_url:
    skip("NATS not configured - set NATS_URL")
else:
    try:
        import nats
        import asyncio
        import json
        
        async def test_nats():
            nc = await nats.connect(nats_url)
            
            # Demo: Simulate multi-agent messaging
            received_messages = []
            
            async def message_handler(msg):
                received_messages.append(json.loads(msg.data.decode()))
            
            # Experiment Tracker subscribes to new papers
            sub = await nc.subscribe("papers.new", cb=message_handler)
            
            # Paper Summarizer publishes a new paper
            paper_event = {"arxiv_id": "2106.09685", "title": "LoRA", "action": "summarized"}
            await nc.publish("papers.new", json.dumps(paper_event).encode())
            
            # Wait for message delivery
            await asyncio.sleep(0.1)
            await sub.unsubscribe()
            await nc.close()
            
            return received_messages
        
        messages = await test_nats()
        
        success(f"Connected to NATS at {nats_url}")
        if messages:
            info(f"Pub/sub test - received: {messages[0]['title']}")
        info("Multi-agent messaging ready")
        
    except Exception as e:
        error(f"NATS connection failed: {e}")

---
## Summary: How It All Fits Together

The Research Assistant uses these services in a coordinated flow:

### RAG Pipeline (notebooks 01)
1. **Ingest**: Load papers from ArXiv
2. **Chunk**: Split into manageable pieces
3. **Embed**: Generate vectors via **LiteLLM**
4. **Store**: Save vectors in **Qdrant**, metadata in **PostgreSQL**
5. **Query**: Search **Qdrant** for relevant chunks
6. **Generate**: Use **LiteLLM** to answer with context
7. **Cache**: Store results in **Valkey**
8. **Trace**: Log everything to **Langfuse**

### Multi-Agent System (notebook 02)
- Agents communicate via **NATS**
- Each agent uses the RAG pipeline
- Coordination happens through pub/sub

### Fine-Tuning (notebook 03)
- Track experiments in **MLflow**
- Train on research domain data
- Deploy improved model via **LiteLLM**

---

**Next**: Continue to `research-assistant/01-langchain-rag.ipynb` to build the RAG pipeline.